In [ ]:
# === Setup ===
# Runtime: ~10 phút trên Colab T4
# Hardware: Cần GPU (T4 là đủ)
import os, random
import numpy as np
import torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
try:
    if 'google.colab' in str(get_ipython()):
        print('Running on Google Colab...')
except NameError:
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        print('Running on Kaggle...')
    else:
        print('Running locally...')

# Reference Solution: Text Classification

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Pipeline hoàn chỉnh (Mã nguồn minh hoạ)
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

texts = ['I love this!', 'You are an idiot!', 'This is nice.', 'Shut up!']
labels = [0, 1, 0, 1]

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts; self.labels = labels; self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=16, return_tensors='pt')
        return {k: v.flatten() for k, v in enc.items() if k != 'token_type_ids'} | {'labels': torch.tensor(self.labels[idx])}

loader = DataLoader(TextDataset(texts, labels, tokenizer), batch_size=2, shuffle=True)
optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):
    for batch in loader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1} Loss: {loss.item():.4f}')